# `gapspipeline_v2` — Process and Verify Clean Segments

Segments the raw L1 data on a **binary** gap mask, then applies the standard
MojitoProcessor pipeline to each clean segment independently.

| Step | Operation |
|------|-----------|
| 1 | Load raw L1 data |
| 2 | Generate binary gap mask (`GapMaskGenerator`, no tapering) |
| 3 | Extract contiguous clean segments at the raw sampling rate |
| 4 | Apply `process_pipeline` to each segment: filter → downsample → trim → window |
| 5 | Plot each segment's periodogram vs the Mojito L1 noise estimate |

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
from lisaglitch import GapMaskGenerator

import MojitoProcessor as mp
from MojitoProcessor.gaps import extract_clean_segments
from MojitoProcessor.process.sigprocess import SignalProcessor, process_pipeline

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

## 1. Configuration

In [ ]:
mojito_data_file = (
    "../Mojito_Data/NOISE_731d_0.25s_L1_source0_0_20251206T220508924302Z.h5"
)

# Set to a float (days) to load only a subset of the file
load_days = None

# ── Gap schedule ──────────────────────────────────────────────────────────────
gap_definitions = {
    "planned": {
        "antenna_repointing": {
            "rate_per_year": 26,
            "duration_hr": 7,
        }
    },
    "unplanned": {},
}

# ── Processing pipeline parameters ───────────────────────────────────────────
downsample_kwargs = {
    "target_fs": 0.2,  # Hz — target sampling rate
    "kaiser_window": 31.0,
}
filter_kwargs = {
    "highpass_cutoff": 5e-5,  # Hz
    "lowpass_cutoff": 0.8 * downsample_kwargs["target_fs"],  # Hz
    "order": 1,
}
trim_kwargs = {
    "fraction": 0.20,  # fraction trimmed from each end after downsampling
}
window_kwargs = {
    "window": "tukey",
    "alpha": 0.025,
}

min_clean_hours = 8.0
max_segment_days = 7.0

## 2. Load Data and Generate Binary Gap Mask

In [ ]:
data = mp.io.load_file(mojito_data_file, load_days=load_days)

print(
    f"Loaded: {data['tdis']['X'].shape[0]:,} samples @ {data['fs']} Hz "
    f"({data['tdis']['X'].shape[0] * data['dt'] / 86400:.2f} days)"
)

gap_gen = GapMaskGenerator(
    sim_t=data["t_tdi"],
    gap_definitions=gap_definitions,
    treat_as_nan=False,
)

binary_mask = gap_gen.generate_mask(
    include_planned=True,
    include_unplanned=False,
).astype(int)

print(
    f"Binary mask: {len(binary_mask):,} samples, "
    f"gap fraction {1 - binary_mask.mean():.4%}"
)

del gap_gen

## 3. Extract Clean Segments at the Raw Rate

Build a raw `SignalProcessor` from the full data, then use `extract_clean_segments`
with the binary mask to cut out contiguous clean stretches.  No filtering or
downsampling yet — we process each segment independently in the next step.

In [ ]:
laser_frequency = float(data["metadata"]["laser_frequency"])

raw_sp = SignalProcessor(
    {ch: data["tdis"][ch] / laser_frequency for ch in ["X", "Y", "Z"]},
    fs=data["fs"],
    t0=float(data["t_tdi"][0]),
)

# Keep noise estimates for plotting; free the large TDI arrays
noise_cov_xyz = data["noise_estimates"]["xyz"]
noise_cov_aet = data["noise_estimates"]["aet"]
del data

raw_segments = extract_clean_segments(
    raw_sp,
    binary_mask,
    min_clean_hours=min_clean_hours,
    max_segment_days=max_segment_days,
)
del raw_sp

print(f"Found {len(raw_segments)} clean segment(s):")
for name, seg in raw_segments.items():
    print(
        f"  {name}: N={seg.N:>8,}, fs={seg.fs} Hz, "
        f"t0={seg.t0/86400:.3f} d, T={seg.T/3600:.1f} h"
    )

## 4. Process Each Segment

Apply the standard MojitoProcessor pipeline to each raw segment independently:
filter → downsample → trim → window.

Each segment is wrapped in a single-entry dict so `process_pipeline` can handle
it directly, and we unpack `"segment0"` from the result.

In [ ]:
processed_segments = {}

for name, seg in raw_segments.items():
    seg.filter(
        low=filter_kwargs["highpass_cutoff"],
        high=filter_kwargs.get("lowpass_cutoff"),
        order=filter_kwargs.get("order", 2),
    )
    seg.downsample(
        downsample_kwargs["target_fs"],
        window=("kaiser", downsample_kwargs.get("kaiser_window", 31.0)),
    )
    seg.trim(fraction=trim_kwargs["fraction"])
    seg.apply_window(
        window=window_kwargs["window"],
        alpha=window_kwargs.get("alpha", 0.025),
    )
    processed_segments[name] = seg

print(f"Processed {len(processed_segments)} segment(s):")
for name, sp in processed_segments.items():
    print(
        f"  {name}: N={sp.N:>7,}, fs={sp.fs} Hz, "
        f"t0={sp.t0/86400:.3f} d, T={sp.T/3600:.1f} h"
    )

## 5. Periodogram vs L1 Estimate — Per Segment

One-sided periodogram:

$$\hat{S}(f_k) = \frac{2\,\Delta t}{N}\left|\tilde{d}(f_k)\right|^2$$

Each row shows one segment (up to 6); left column = XYZ, right = AET.
The red dashed line is the Mojito L1 noise estimate.  A good result shows the
periodogram lying on top of the L1 curve throughout the science band (green).

In [ ]:
# ── L1 noise model ────────────────────────────────────────────────────────────
n_noise_bins = noise_cov_xyz[0].shape[0]
noise_freqs = np.logspace(-5, 0, n_noise_bins)

l1_xyz = {
    ch: noise_cov_xyz[0][:, i, i] / laser_frequency**2
    for i, ch in enumerate(["X", "Y", "Z"])
}
l1_aet = {
    ch: noise_cov_aet[0][:, i, i] / laser_frequency**2
    for i, ch in enumerate(["A", "E", "T"])
}

# ── Per-segment periodogram grid ──────────────────────────────────────────────
MAX_PLOT_SEGMENTS = 6
seg_names = list(processed_segments.keys())[:MAX_PLOT_SEGMENTS]
n_plot = len(seg_names)

xyz_colors = ["C0", "C1", "C2"]
aet_colors = ["#e377c2", "#9467bd", "#8c564b"]

fig, axes = plt.subplots(
    n_plot,
    2,
    figsize=(16, 4 * n_plot),
    sharex=False,
    sharey=True,
    squeeze=False,
)

for row, name in enumerate(seg_names):
    sp = processed_segments[name]
    sp_aet = sp.to_aet()

    freq, fft_xyz = sp.fft()
    _, fft_aet = sp_aet.fft()
    psd_norm = 2.0 * sp.dt / sp.N
    nyquist = sp.fs / 2.0

    # XYZ — data channels vs single red L1 estimate
    ax = axes[row, 0]
    for i, ch in enumerate(["X", "Y", "Z"]):
        psd = np.abs(fft_xyz[ch]) ** 2 * psd_norm
        ax.loglog(freq[1:], psd[1:], lw=0.8, color=xyz_colors[i], label=f"TDI-{ch}")
    # Single red dashed L1 (use X channel as representative; all three are plotted)
    for i, ch in enumerate(["X", "Y", "Z"]):
        label = "Mojito L1" if i == 0 else None
        ax.loglog(noise_freqs, l1_xyz[ch], ls="--", lw=2.0, color="red", label=label)
    ax.axvspan(1e-4, nyquist, alpha=0.12, color="green", label="Science band")
    ax.set_xlim(1e-5, nyquist)
    ax.set_ylim(1e-53, 1e-35)
    ax.set_ylabel("PSD [1/Hz]", fontsize=16)
    ax.set_title(
        f"{name}  |  t0={sp.t0/86400:.3f} d,  T={sp.T/86400:.2f} d,  N={sp.N:,}",
        fontsize=14,
        fontweight="bold",
    )
    ax.legend(fontsize=11, loc="upper left", framealpha=0.95)
    ax.grid(True, which="both", alpha=0.3)
    ax.tick_params(axis="both", which="major", labelsize=14)

    # AET — data channels vs single red L1 estimate
    ax = axes[row, 1]
    for i, ch in enumerate(["A", "E", "T"]):
        psd = np.abs(fft_aet[ch]) ** 2 * psd_norm
        ax.loglog(freq[1:], psd[1:], lw=0.8, color=aet_colors[i], label=f"TDI-{ch}")
    for i, ch in enumerate(["A", "E", "T"]):
        label = "Mojito L1" if i == 0 else None
        ax.loglog(noise_freqs, l1_aet[ch], ls="--", lw=2.0, color="red", label=label)
    ax.axvspan(1e-4, nyquist, alpha=0.12, color="green")
    ax.set_xlim(1e-5, nyquist)
    ax.set_ylim(1e-53, 1e-35)
    ax.legend(fontsize=11, loc="upper left", framealpha=0.95)
    ax.grid(True, which="both", alpha=0.3)
    ax.tick_params(axis="both", which="major", labelsize=14)

axes[-1, 0].set_xlabel("Frequency [Hz]", fontsize=16)
axes[-1, 1].set_xlabel("Frequency [Hz]", fontsize=16)

fig.suptitle(
    f"gapspipeline v2 — Periodogram vs L1 Estimate (first {n_plot} segments)",
    fontsize=16,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Data-loss report ─────────────────────────────────────────────────────────
# Raw cadence is 0.25 s (4 Hz), encoded in the data file name.
dt_raw = 0.25  # seconds

total_samples = len(binary_mask)
T_total = total_samples * dt_raw  # s — full mission duration

# ── Gap-related loss (mask = 0 AND clean stretches too short to keep) ─────────
clean_samples = int(binary_mask.sum())
T_clean_mask = clean_samples * dt_raw  # s — time where mask = 1
T_mask_zeros = T_total - T_clean_mask  # s — time under gap mask

# Back-calculate total pre-trim (post-downsample) segment duration.
# trim(fraction=f) removes int(N*f/2) from each end → kept ≈ N*(1-f).
# Inverse: T_pretrim ≈ T_proc / (1 - f).
frac = trim_kwargs["fraction"]
T_proc_total = sum(sp.T for sp in processed_segments.values())
T_pretrim_total = T_proc_total / (1.0 - frac)

# Clean stretches discarded as too short (< min_clean_hours) are also
# caused by gaps surrounding them, so they belong in the "gap" bucket.
T_short_seg = T_clean_mask - T_pretrim_total  # s — discarded short segs
T_gap_total = T_mask_zeros + T_short_seg  # (1): total gap-related loss

# ── Trim loss ─────────────────────────────────────────────────────────────────
T_trim_loss = T_pretrim_total - T_proc_total  # (2): from trim() call only

# ── Combined ──────────────────────────────────────────────────────────────────
T_total_loss = T_gap_total + T_trim_loss  # (3) = (1) + (2)
# Cross-check: should equal T_total - T_proc_total
assert abs(T_total_loss - (T_total - T_proc_total)) < 1.0, "accounting error"


# ── Print report ──────────────────────────────────────────────────────────────
def _r(s, ref=T_total):
    return f"{s/86400:7.2f} d  ({s/3600:8.1f} h)  [{s/ref*100:5.2f}% of total]"


W = 72
print("=" * W)
print(
    f"  TOTAL data in file              {T_total/86400:.2f} d  ({T_total/3600:.1f} h)"
)
print("=" * W)
print(f"(1) Loss due to gaps              {_r(T_gap_total)}")
print(f"    of which  — mask = 0          {_r(T_mask_zeros)}")
print(f"              — short segs (< {min_clean_hours:.0f} h) {_r(T_short_seg)}")
print(f"(2) Loss due to trimming          {_r(T_trim_loss)}")
print(f"    (trim fraction = {frac:.0%} per segment)")
print(f"(3) Loss due to gaps + trimming   {_r(T_total_loss)}")
print("-" * W)
print(f"    Remaining (processed)         {_r(T_proc_total)}")
print("=" * W)